In [ ]:
# =========================================================
# 위성점수 계산 + 영입제한 / 위성(Satellite) / 기타 분류 코드
# =========================================================

import numpy as np
import pandas as pd

# df에 메인 DB 혹은 다른 세그먼트 점수 부여한 df 넣으시면 되요
# df = pd.read_csv("")


# =========================================================
# 0. 설정값
# =========================================================

# ---------------------------------------------------------
# 0-1. 위성점수 가중치
# ---------------------------------------------------------
WEIGHT_DONATION = 0.30
WEIGHT_PEAK_CHAT = 0.30
WEIGHT_AVG_VIEWER = 0.30
WEIGHT_FANDOM = 0.10

NORMALIZE_WEIGHTS = True

# 극단값 완화용 분위수 클리핑
USE_WINSORIZE = True
LOWER_Q = 0.001
UPPER_Q = 0.999

# 팬덤지수는 음수도 있으므로 log 변환하지 않고 min-max만 적용
USE_LOG_FOR_FANDOM = False

FILL_NA_VALUE = 0.0


# ---------------------------------------------------------
# 0-2. 영입제한 기준
# ---------------------------------------------------------
# 의미:
# - 지표가 너무 강하거나 팔로워 체급이 커서
#   매력도는 높지만 실제 영입 난이도가 높은 그룹

RESTRICT_DONATION_QUANTILE = 0.95       # 도네이션 상위 5%
RESTRICT_PEAK_CHAT_QUANTILE = 0.95      # 6분 최고채팅 상위 5%
RESTRICT_AVG_VIEWER_CUTOFF = 10000      # 평균 시청자 최댓값 기준

USE_RESTRICT_SCORE_QUANTILE = False
RESTRICT_SCORE_QUANTILE = 0.90
RESTRICT_SCORE_CUTOFF = 0.865

RESTRICT_SOLO_TYPES = ["솔로확정"]

# 팔로워 기반 영입제한 기준
RESTRICT_FOLLOWER_CUTOFF = 40000
RESTRICT_FOLLOWER_REQUIRE_SOLO = True
RESTRICT_FOLLOWER_SOLO_TYPES = ["솔로확정", "솔로추정"]


# ---------------------------------------------------------
# 0-3. 위성(Satellite) 기준
# ---------------------------------------------------------
# 의미:
# - 영입제한만큼 초고체급은 아니지만,
#   실질적으로 영입 검토할 만한 개인형 고화력 후보

SATELLITE_DONATION_QUANTILE = 0.85      # 도네이션 상위 15%
SATELLITE_PEAK_CHAT_QUANTILE = 0.85     # 6분 최고채팅 상위 15%
SATELLITE_AVG_VIEWER_CUTOFF = 1000      # 평균 시청자 최댓값 기준

USE_SATELLITE_SCORE_QUANTILE = True
SATELLITE_SCORE_QUANTILE = 0.85
SATELLITE_SCORE_CUTOFF = 0.65

SATELLITE_SOLO_TYPES = ["솔로확정", "솔로추정"]


# =========================================================
# 1. 유틸 함수
# =========================================================

def normalize_weights(weight_dict):
    total = sum(weight_dict.values())
    if total == 0:
        raise ValueError("가중치 합이 0입니다.")
    return {k: v / total for k, v in weight_dict.items()}


def robust_minmax(series, use_log=True, use_winsorize=True, lower_q=0.01, upper_q=0.99):
    """
    series를 0~1로 변환.
    - use_log=True면 log1p 적용
    - use_winsorize=True면 상하위 분위수 기준으로 클리핑
    """
    s = pd.to_numeric(series, errors="coerce")

    if use_log:
        s = s.clip(lower=0)
        s = np.log1p(s)

    if use_winsorize:
        lo = s.quantile(lower_q)
        hi = s.quantile(upper_q)
        s = s.clip(lower=lo, upper=hi)

    min_v = s.min()
    max_v = s.max()

    if pd.isna(min_v) or pd.isna(max_v) or max_v == min_v:
        return pd.Series(FILL_NA_VALUE, index=series.index)

    return ((s - min_v) / (max_v - min_v)).fillna(FILL_NA_VALUE)


def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


# =========================================================
# 2. 컬럼 매핑
# =========================================================

col_donation = first_existing(df, ["도네이션", "donation", "donation_revenue"])
col_peak_chat = first_existing(df, ["6분_최고채팅", "6분최고채팅", "peak_chat_6min"])
col_avg_viewer = first_existing(df, ["평균_시청자_최댓값", "평균시청자_최댓값", "avg_viewer_max"])
col_fandom = first_existing(df, ["팬덤지수", "fandom_index"])
col_followers = first_existing(df, ["최고_팔로워", "최고팔로워", "followers", "max_followers"])

required_cols = {
    "도네이션": col_donation,
    "6분_최고채팅": col_peak_chat,
    "평균_시청자_최댓값": col_avg_viewer,
}

missing = [k for k, v in required_cols.items() if v is None]
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

if "솔로성분류" not in df.columns:
    raise ValueError("필수 컬럼이 없습니다: 솔로성분류")


print("[컬럼 매핑]")
print("도네이션:", col_donation)
print("6분_최고채팅:", col_peak_chat)
print("평균_시청자_최댓값:", col_avg_viewer)
print("팬덤지수:", col_fandom)
print("최고_팔로워:", col_followers)


# =========================================================
# 3. log-minmax 기반 지표 생성
# =========================================================

df["도네이션_log_minmax"] = robust_minmax(
    df[col_donation],
    use_log=True,
    use_winsorize=USE_WINSORIZE,
    lower_q=LOWER_Q,
    upper_q=UPPER_Q,
)

df["6분_최고채팅_log_minmax"] = robust_minmax(
    df[col_peak_chat],
    use_log=True,
    use_winsorize=USE_WINSORIZE,
    lower_q=LOWER_Q,
    upper_q=UPPER_Q,
)

df["평균_시청자_최댓값_log_minmax"] = robust_minmax(
    df[col_avg_viewer],
    use_log=True,
    use_winsorize=USE_WINSORIZE,
    lower_q=LOWER_Q,
    upper_q=UPPER_Q,
)

if col_fandom is not None:
    df["팬덤지수_minmax"] = robust_minmax(
        df[col_fandom],
        use_log=USE_LOG_FOR_FANDOM,
        use_winsorize=USE_WINSORIZE,
        lower_q=LOWER_Q,
        upper_q=UPPER_Q,
    )
else:
    df["팬덤지수_minmax"] = 0.5


# =========================================================
# 4. 위성점수 계산
# =========================================================

weights = {
    "donation": WEIGHT_DONATION,
    "peak_chat": WEIGHT_PEAK_CHAT,
    "avg_viewer": WEIGHT_AVG_VIEWER,
    "fandom": WEIGHT_FANDOM,
}

if NORMALIZE_WEIGHTS:
    weights = normalize_weights(weights)

print("\n[사용 가중치]")
print(weights)

df["위성점수_log_minmax_raw"] = (
    df["도네이션_log_minmax"] * weights["donation"]
    + df["6분_최고채팅_log_minmax"] * weights["peak_chat"]
    + df["평균_시청자_최댓값_log_minmax"] * weights["avg_viewer"]
    + df["팬덤지수_minmax"] * weights["fandom"]
)

# 솔로성보정계수가 있으면 반영
if "솔로성보정계수" in df.columns:
    df["솔로성보정계수"] = pd.to_numeric(df["솔로성보정계수"], errors="coerce").fillna(1)
    df["위성점수_log_minmax"] = df["위성점수_log_minmax_raw"] * df["솔로성보정계수"]
else:
    df["위성점수_log_minmax"] = df["위성점수_log_minmax_raw"]


# =========================================================
# 5. 영입제한 / 위성(Satellite) 기준값 계산
# =========================================================

df[col_donation] = pd.to_numeric(df[col_donation], errors="coerce")
df[col_peak_chat] = pd.to_numeric(df[col_peak_chat], errors="coerce")
df[col_avg_viewer] = pd.to_numeric(df[col_avg_viewer], errors="coerce")

if col_followers is not None:
    df[col_followers] = pd.to_numeric(df[col_followers], errors="coerce")

restrict_donation_cutoff = df[col_donation].quantile(RESTRICT_DONATION_QUANTILE)
restrict_peak_chat_cutoff = df[col_peak_chat].quantile(RESTRICT_PEAK_CHAT_QUANTILE)
restrict_avg_viewer_cutoff = RESTRICT_AVG_VIEWER_CUTOFF

satellite_donation_cutoff = df[col_donation].quantile(SATELLITE_DONATION_QUANTILE)
satellite_peak_chat_cutoff = df[col_peak_chat].quantile(SATELLITE_PEAK_CHAT_QUANTILE)
satellite_avg_viewer_cutoff = SATELLITE_AVG_VIEWER_CUTOFF

if USE_RESTRICT_SCORE_QUANTILE:
    restrict_score_cutoff = df["위성점수_log_minmax"].quantile(RESTRICT_SCORE_QUANTILE)
else:
    restrict_score_cutoff = RESTRICT_SCORE_CUTOFF

if USE_SATELLITE_SCORE_QUANTILE:
    satellite_score_cutoff = df["위성점수_log_minmax"].quantile(SATELLITE_SCORE_QUANTILE)
else:
    satellite_score_cutoff = SATELLITE_SCORE_CUTOFF


print("\n" + "=" * 80)
print("[영입제한 / 위성(Satellite) 분류 기준]")
print("=" * 80)
print(f"영입제한 도네이션 기준: {restrict_donation_cutoff:,.0f}")
print(f"영입제한 6분 최고채팅 기준: {restrict_peak_chat_cutoff:,.0f}")
print(f"영입제한 평균 시청자 기준: {restrict_avg_viewer_cutoff:,.0f}")
print(f"영입제한 위성점수 기준: {restrict_score_cutoff:.4f}")
print(f"영입제한 최고 팔로워 기준: {RESTRICT_FOLLOWER_CUTOFF:,.0f}")

print("-" * 80)
print(f"위성 도네이션 기준: {satellite_donation_cutoff:,.0f}")
print(f"위성 6분 최고채팅 기준: {satellite_peak_chat_cutoff:,.0f}")
print(f"위성 평균 시청자 기준: {satellite_avg_viewer_cutoff:,.0f}")
print(f"위성점수 기준: {satellite_score_cutoff:.4f}")


# =========================================================
# 6. 조건 컬럼 생성
# =========================================================

# ---------------------------------------------------------
# 6-1. 영입제한: 지표 기반
# ---------------------------------------------------------
df["조건_영입제한_솔로성충족"] = df["솔로성분류"].isin(RESTRICT_SOLO_TYPES)
df["조건_영입제한_도네이션충족"] = df[col_donation] >= restrict_donation_cutoff
df["조건_영입제한_채팅충족"] = df[col_peak_chat] >= restrict_peak_chat_cutoff
df["조건_영입제한_평균시청자충족"] = df[col_avg_viewer] >= restrict_avg_viewer_cutoff
df["조건_영입제한_점수충족"] = df["위성점수_log_minmax"] >= restrict_score_cutoff

df["조건_영입제한_지표기반"] = (
    df["조건_영입제한_솔로성충족"]
    & df["조건_영입제한_도네이션충족"]
    & df["조건_영입제한_채팅충족"]
    & df["조건_영입제한_평균시청자충족"]
    & df["조건_영입제한_점수충족"]
)


# ---------------------------------------------------------
# 6-2. 영입제한: 팔로워 기반
# ---------------------------------------------------------
if col_followers is not None:
    df["조건_영입제한_팔로워충족"] = df[col_followers] >= RESTRICT_FOLLOWER_CUTOFF
else:
    df["조건_영입제한_팔로워충족"] = False

if RESTRICT_FOLLOWER_REQUIRE_SOLO:
    df["조건_영입제한_팔로워기반"] = (
        df["조건_영입제한_팔로워충족"]
        & df["조건_영입제한_점수충족"]
        & df["솔로성분류"].isin(RESTRICT_FOLLOWER_SOLO_TYPES)
    )
else:
    df["조건_영입제한_팔로워기반"] = (
        df["조건_영입제한_팔로워충족"]
        & df["조건_영입제한_점수충족"]
    )


# ---------------------------------------------------------
# 6-3. 위성(Satellite)
# ---------------------------------------------------------
df["조건_위성_솔로성충족"] = df["솔로성분류"].isin(SATELLITE_SOLO_TYPES)
df["조건_위성_도네이션충족"] = df[col_donation] >= satellite_donation_cutoff
df["조건_위성_채팅충족"] = df[col_peak_chat] >= satellite_peak_chat_cutoff
df["조건_위성_평균시청자충족"] = df[col_avg_viewer] >= satellite_avg_viewer_cutoff
df["조건_위성_점수충족"] = df["위성점수_log_minmax"] >= satellite_score_cutoff


# =========================================================
# 7. 최종 세그먼트 분류
# =========================================================

df["영입제한여부"] = (
    df["조건_영입제한_지표기반"]
    | df["조건_영입제한_팔로워기반"]
)

df["위성여부"] = (
    ~df["영입제한여부"]
    & df["조건_위성_솔로성충족"]
    & df["조건_위성_도네이션충족"]
    & df["조건_위성_채팅충족"]
    & df["조건_위성_평균시청자충족"]
    & df["조건_위성_점수충족"]
)


def assign_satellite_segment(row):
    if row["영입제한여부"]:
        return "영입제한"
    if row["위성여부"]:
        return "위성(Satellite)"
    return "기타"


df["세그먼트_위성"] = df.apply(assign_satellite_segment, axis=1)

# =========================================================
# 9. 최종 산출용 컬럼 정리
# - 계산용 log-minmax/조건 컬럼은 제거
# - 최종 분석에 필요한 위성점수와 세그먼트_위성은 유지
# =========================================================

# 100점 기준 최종 점수
df["위성점수"] = (df["위성점수_log_minmax"].clip(0, 1) * 100).round(2)

# 계산용 파생컬럼 제거
drop_cols = [
    # log-minmax 중간 파생변수
    "도네이션_log_minmax",
    "6분_최고채팅_log_minmax",
    "평균_시청자_최댓값_log_minmax",
    "팬덤지수_minmax",
    "위성점수_log_minmax_raw",
    "위성점수_log_minmax",

    # 영입제한 조건 판정용 중간 컬럼
    "조건_영입제한_솔로성충족",
    "조건_영입제한_도네이션충족",
    "조건_영입제한_채팅충족",
    "조건_영입제한_평균시청자충족",
    "조건_영입제한_점수충족",
    "조건_영입제한_지표기반",
    "조건_영입제한_팔로워충족",
    "조건_영입제한_팔로워기반",

    # 위성 조건 판정용 중간 컬럼
    "조건_위성_솔로성충족",
    "조건_위성_도네이션충족",
    "조건_위성_채팅충족",
    "조건_위성_평균시청자충족",
    "조건_위성_점수충족",

    # 내부 bool 판정 컬럼
    "영입제한여부",
    "위성여부",
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])